# Week 3 — Data Preprocessing

## Objectives

- Load and combine the monthly CRMLS sold-property files.
- Restrict the data to residential single-family properties.
- Prepare numerical and categorical variables for modeling.
- Create missing-value indicators.
- Use the most recent month as the test set and the preceding five months as the training set.
- Fit all preprocessing steps using the training data only.
- Save the cleaned datasets inside `results/02/`.

## Required input

Place the raw monthly CSV files inside:

`data/`

The expected files are the monthly CRMLS sold-property files, such as files whose names begin with `CRMLSSold`.

## Generated files

This notebook creates only two files:

- `results/02/cleaned_preprocessed_data.csv`
- `results/02/cleaned_unencoded_data.csv`

The encoded file is the required input for `03_baseline_model.ipynb` and `04_model_comparison.ipynb`.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import sparse

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## 1. Define the project folders and locate the raw files

The notebook works when it is stored either:

- in the project root folder; or
- inside a `notebooks` folder.

All generated files are placed inside `results/02/`.


In [2]:
working_dir = Path.cwd()

project_root = (
    working_dir.parent
    if working_dir.name.lower() == "notebooks"
    else working_dir
)

data_dir = project_root / "data"
results_dir = project_root / "results" / "02"
results_dir.mkdir(parents=True, exist_ok=True)

if not data_dir.exists():
    raise FileNotFoundError(
        "The data folder was not found. Create a folder named 'data' "
        "in the project root and place the monthly raw CSV files inside it."
    )

preferred_files = sorted(data_dir.glob("CRMLSSold*.csv"))

if preferred_files:
    csv_files = preferred_files
else:
    excluded_output_names = {
        "cleaned_preprocessed_data.csv",
        "cleaned_unencoded_data.csv",
    }

    csv_files = [
        path
        for path in sorted(data_dir.glob("*.csv"))
        if path.name not in excluded_output_names
    ]

if not csv_files:
    raise FileNotFoundError(
        "No raw monthly CSV files were found inside the data folder."
    )

print("Number of raw CSV files found:", len(csv_files))
print("Output folder: results/02/")

if len(csv_files) != 6:
    print(
        "Warning: The original project expected six monthly files, "
        f"but {len(csv_files)} file(s) were found."
    )

pd.DataFrame({
    "Source_File": [path.name for path in csv_files],
    "File_Size_MB": [
        round(path.stat().st_size / 1024**2, 2)
        for path in csv_files
    ],
})


Number of raw CSV files found: 6
Output folder: results/02/


,Source_File,File_Size_MB
0,CRMLSSold202512.csv,10.8900
1,CRMLSSold202601.csv,8.7400
2,CRMLSSold202602.csv,9.6000
3,CRMLSSold202603.csv,11.9400
4,CRMLSSold202604.csv,12.3600
5,CRMLSSold202605.csv,12.3000


## 2. Load and combine the monthly files

A `SourceFile` column is added so every observation can be traced to its original monthly file.


In [3]:
monthly_frames = []
file_summary = []

for file in csv_files:
    monthly_df = pd.read_csv(file, low_memory=False)
    monthly_df.columns = monthly_df.columns.str.strip()

    duplicate_columns = (
        monthly_df.columns[monthly_df.columns.duplicated()]
        .tolist()
    )

    if duplicate_columns:
        raise ValueError(
            f"Duplicate columns were found in {file.name}: "
            f"{duplicate_columns[:10]}"
        )

    monthly_df["SourceFile"] = file.name
    monthly_frames.append(monthly_df)

    file_summary.append({
        "Source_File": file.name,
        "Rows": monthly_df.shape[0],
        "Columns": monthly_df.shape[1],
    })

df = pd.concat(
    monthly_frames,
    ignore_index=True,
    sort=False,
)

print("Combined dataset shape:", df.shape)
pd.DataFrame(file_summary)


Combined dataset shape: (124404, 79)


,Source_File,Rows,Columns
0,CRMLSSold202512.csv,20538,79
1,CRMLSSold202601.csv,16487,79
2,CRMLSSold202602.csv,18124,79
3,CRMLSSold202603.csv,22583,79
4,CRMLSSold202604.csv,23412,79
5,CRMLSSold202605.csv,23260,79


## 3. Standardize column names and validate required columns

The original MLS names are renamed to the shorter names used in the later notebooks.


In [4]:
df.columns = df.columns.str.strip()

rename_map = {
    "BedroomsTotal": "Bedrooms",
    "BathroomsTotalInteger": "Bathrooms",
    "LotSizeSquareFeet": "LotSize",
}

df = df.rename(
    columns={
        old_name: new_name
        for old_name, new_name in rename_map.items()
        if old_name in df.columns
        and new_name not in df.columns
    }
)

target_col = "ClosePrice"

numeric_features = [
    "LivingArea",
    "Bedrooms",
    "Bathrooms",
    "LotSize",
    "YearBuilt",
]

categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
]

required_columns = (
    [target_col]
    + numeric_features
    + categorical_features
    + [
        "PropertyType",
        "PropertySubType",
        "SourceFile",
    ]
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

duplicate_columns = df.columns[df.columns.duplicated()].tolist()

if duplicate_columns:
    raise ValueError(
        f"Duplicate column names were found: "
        f"{duplicate_columns[:10]}"
    )

print("All required columns are available.")


All required columns are available.


## 4. Create the month variable from each source filename

The six-digit `YYYYMM` value in each filename is converted into a monthly date.


In [5]:
df["YearMonth"] = (
    df["SourceFile"]
    .astype("string")
    .str.extract(r"(\d{6})", expand=False)
)

df["YearMonth"] = pd.to_datetime(
    df["YearMonth"],
    format="%Y%m",
    errors="coerce",
)

missing_month_rows = int(df["YearMonth"].isna().sum())

if missing_month_rows:
    invalid_files = (
        df.loc[df["YearMonth"].isna(), "SourceFile"]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        "A YYYYMM month could not be extracted from these filenames: "
        f"{invalid_files}"
    )

month_summary = (
    df[["SourceFile", "YearMonth"]]
    .drop_duplicates()
    .sort_values("YearMonth")
    .reset_index(drop=True)
)

month_summary


,SourceFile,YearMonth
0,CRMLSSold202512.csv,2025-12-01
1,CRMLSSold202601.csv,2026-01-01
2,CRMLSSold202602.csv,2026-02-01
3,CRMLSSold202603.csv,2026-03-01
4,CRMLSSold202604.csv,2026-04-01
5,CRMLSSold202605.csv,2026-05-01


## 5. Filter to residential single-family properties

The analysis retains observations satisfying both conditions:

- `PropertyType = Residential`
- `PropertySubType = SingleFamilyResidence`


In [6]:
property_type = (
    df["PropertyType"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

property_subtype = (
    df["PropertySubType"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

single_family_mask = (
    property_type.eq("residential")
    & property_subtype.eq("singlefamilyresidence")
)

df_filtered = df.loc[single_family_mask].copy()

if df_filtered.empty:
    raise ValueError(
        "No residential single-family observations remained "
        "after filtering."
    )

filter_summary = pd.DataFrame({
    "Dataset": [
        "Combined raw data",
        "Residential single-family data",
    ],
    "Rows": [
        len(df),
        len(df_filtered),
    ],
})

filter_summary["Share_of_Raw_Data"] = (
    filter_summary["Rows"] / len(df)
)

filter_summary


,Dataset,Rows,Share_of_Raw_Data
0,Combined raw data,124404,1.0000
1,Residential single-family data,61727,0.4962


## 6. Select and convert the modeling variables

Numerical values are cleaned and converted with `errors="coerce"`, so invalid entries become missing values. Categorical values are standardized as trimmed strings.


In [7]:
model_columns = (
    [target_col]
    + numeric_features
    + categorical_features
    + ["YearMonth"]
)

model_df = df_filtered[model_columns].copy()

for column in [target_col] + numeric_features:
    cleaned_values = (
        model_df[column]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

    model_df[column] = pd.to_numeric(
        cleaned_values,
        errors="coerce",
    )

for column in categorical_features:
    model_df[column] = model_df[column].apply(
        lambda value: (
            str(value).strip()
            if pd.notna(value)
            else np.nan
        )
    )

print("Modeling dataset shape:", model_df.shape)
model_df.head()


Modeling dataset shape: (61727, 10)


,ClosePrice,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,CountyOrParish,City,PostalCode,YearMonth
0,"1,998,000.0000","2,045.0000",4.0000,2.0000,"10,080.0000","1,968.0000",Contra Costa,Walnut Creek,94596,2025-12-01
2,"2,214,421.0000","3,050.0000",4.0000,4.0000,"34,745.0000","1,957.0000",Los Angeles,Woodland Hills,91364,2025-12-01
3,"1,200,000.0000","1,594.0000",4.0000,2.0000,"6,600.0000","1,978.0000",Santa Clara,San Jose,95121,2025-12-01
7,"3,100,000.0000","2,700.0000",5.0000,3.0000,"8,262.0000","2,025.0000",Santa Clara,San Jose,95124,2025-12-01
9,"2,900,000.0000","2,948.0000",5.0000,4.0000,"9,222.0000","2,023.0000",Santa Clara,San Jose,95128,2025-12-01


## 7. Review missing values before preprocessing

This table is displayed for documentation. Missing numerical predictors are handled by median imputation, and missing categorical predictors are replaced with `"Missing"`.


In [8]:
missing_summary = pd.DataFrame({
    "Missing_Count": model_df.isna().sum(),
    "Missing_Percent": model_df.isna().mean() * 100,
})

missing_summary.index.name = "Variable"

missing_summary.sort_values(
    "Missing_Percent",
    ascending=False,
)


,Missing_Count,Missing_Percent
Variable,,
LotSize,1081,1.7513
YearBuilt,36,0.0583
LivingArea,30,0.0486
City,14,0.0227
Bathrooms,1,0.0016
PostalCode,1,0.0016
ClosePrice,0,0.0000
Bedrooms,0,0.0000
CountyOrParish,0,0.0000


## 8. Remove observations with a missing target

Rows with a missing `ClosePrice` cannot be used for supervised model training. Predictor missing values are retained for pipeline-based imputation.


In [9]:
rows_before_target_filter = len(model_df)

model_df = (
    model_df
    .dropna(subset=[target_col])
    .copy()
)

rows_removed = (
    rows_before_target_filter
    - len(model_df)
)

print(
    "Rows removed because ClosePrice was missing:",
    rows_removed,
)
print(
    "Remaining observations:",
    len(model_df),
)


Rows removed because ClosePrice was missing: 0
Remaining observations: 61727


## 9. Create numerical missing-value indicators

For each numerical predictor, an additional binary feature records whether the original value was missing.


In [10]:
missing_flag_features = []

for column in numeric_features:
    flag_column = f"{column}_missing"
    model_df[flag_column] = (
        model_df[column]
        .isna()
        .astype(np.int8)
    )
    missing_flag_features.append(flag_column)

print(
    "Missing-value indicator columns:",
    missing_flag_features,
)


Missing-value indicator columns: ['LivingArea_missing', 'Bedrooms_missing', 'Bathrooms_missing', 'LotSize_missing', 'YearBuilt_missing']


## 10. Create the chronological training and test sets

The most recent month is used as the test set. The five immediately preceding months are used as the training set.

When fewer than six valid months are available, all months before the most recent month are used for training and a warning is displayed.


In [11]:
available_months = sorted(
    model_df["YearMonth"]
    .dropna()
    .unique()
)

if len(available_months) < 2:
    raise ValueError(
        "At least two distinct months are required "
        "for a chronological train/test split."
    )

test_month = available_months[-1]
training_month_count = min(
    5,
    len(available_months) - 1,
)

train_months = available_months[
    -(training_month_count + 1):-1
]

if training_month_count < 5:
    print(
        "Warning: Fewer than five training months "
        "were available."
    )

train_df = model_df.loc[
    model_df["YearMonth"].isin(train_months)
].copy()

test_df = model_df.loc[
    model_df["YearMonth"].eq(test_month)
].copy()

if train_df.empty or test_df.empty:
    raise ValueError(
        "The chronological split produced an empty "
        "training or test set."
    )

split_summary = pd.DataFrame({
    "Split": ["Train", "Test"],
    "Months": [
        ", ".join(
            pd.Timestamp(month).strftime("%Y-%m")
            for month in train_months
        ),
        pd.Timestamp(test_month).strftime("%Y-%m"),
    ],
    "Rows": [
        len(train_df),
        len(test_df),
    ],
})

split_summary


,Split,Months,Rows
0,Train,"2025-12, 2026-01, 2026-02, 2026-03, 2026-04",49703
1,Test,2026-05,12024


## 11. Separate predictors and target

The predictor set contains:

- five numerical variables;
- three categorical variables;
- five numerical missing-value indicators.


In [12]:
feature_columns = (
    numeric_features
    + categorical_features
    + missing_flag_features
)

X_train = train_df[feature_columns].copy()
y_train = train_df[target_col].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_col].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Number of input features:", len(feature_columns))


X_train shape: (49703, 13)
X_test shape: (12024, 13)
Number of input features: 13


## 12. Build the preprocessing pipeline

### Numerical variables

- Fill missing values using the training-set median.
- Standardize variables with `StandardScaler`.

### Categorical variables

- Replace missing values with `"Missing"`.
- Apply one-hot encoding.
- Ignore categories that appear only in the test set.

The pipeline is fitted using the training data only to prevent information leakage.


In [13]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
            + missing_flag_features,
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features,
        ),
    ],
    sparse_threshold=1.0,
)

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,1.0
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## 13. Fit the preprocessing pipeline

The training data are used to estimate medians, means, standard deviations, and categorical levels. The fitted pipeline is then applied unchanged to the test data.


In [14]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(
    "Processed training shape:",
    X_train_processed.shape,
)
print(
    "Processed test shape:",
    X_test_processed.shape,
)
print(
    "Training matrix is sparse:",
    sparse.issparse(X_train_processed),
)


Processed training shape: (49703, 2447)
Processed test shape: (12024, 2447)
Training matrix is sparse: True


## 14. Recover the processed feature names

The numerical columns retain their original names. One-hot encoded columns receive category-specific names.


In [15]:
numeric_output_features = (
    numeric_features
    + missing_flag_features
)

categorical_output_features = (
    preprocessor
    .named_transformers_["cat"]
    .named_steps["encoder"]
    .get_feature_names_out(
        categorical_features
    )
)

processed_feature_names = (
    list(numeric_output_features)
    + list(categorical_output_features)
)

expected_feature_count = (
    X_train_processed.shape[1]
)

if len(processed_feature_names) != expected_feature_count:
    raise ValueError(
        "The number of processed feature names "
        "does not match the processed matrix."
    )

print(
    "Number of processed features:",
    len(processed_feature_names),
)


Number of processed features: 2447


## 15. Convert the processed matrices to DataFrames

Sparse DataFrames are used when possible to reduce memory usage before the CSV files are written.


In [16]:
def processed_matrix_to_dataframe(
    matrix,
    index,
    columns,
):
    if sparse.issparse(matrix):
        return pd.DataFrame.sparse.from_spmatrix(
            matrix.astype(np.float32),
            index=index,
            columns=columns,
        )

    return pd.DataFrame(
        np.asarray(
            matrix,
            dtype=np.float32,
        ),
        index=index,
        columns=columns,
    )


X_train_processed_df = (
    processed_matrix_to_dataframe(
        X_train_processed,
        X_train.index,
        processed_feature_names,
    )
)

X_test_processed_df = (
    processed_matrix_to_dataframe(
        X_test_processed,
        X_test.index,
        processed_feature_names,
    )
)

X_train_processed_df.head()


,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,LivingArea_missing,Bedrooms_missing,Bathrooms_missing,LotSize_missing,YearBuilt_missing,CountyOrParish_Alameda,CountyOrParish_Amador,CountyOrParish_Butte,CountyOrParish_Calaveras,CountyOrParish_Colusa,CountyOrParish_Contra Costa,CountyOrParish_El Dorado,CountyOrParish_Foreign Country,CountyOrParish_Fresno,CountyOrParish_Glenn,CountyOrParish_Humboldt,CountyOrParish_Imperial,CountyOrParish_Inyo,CountyOrParish_Kern,CountyOrParish_Kings,CountyOrParish_Lake,CountyOrParish_Lassen,CountyOrParish_Los Angeles,CountyOrParish_Madera,CountyOrParish_Marin,CountyOrParish_Mariposa,CountyOrParish_Mendocino,CountyOrParish_Merced,CountyOrParish_Modoc,CountyOrParish_Mono,CountyOrParish_Monterey,CountyOrParish_Napa,CountyOrParish_Nevada,CountyOrParish_Orange,CountyOrParish_Other,CountyOrParish_Other State,CountyOrParish_Placer,CountyOrParish_Plumas,CountyOrParish_Riverside,CountyOrParish_Sacramento,CountyOrParish_San Benito,CountyOrParish_San Bernardino,CountyOrParish_San Diego,CountyOrParish_San Francisco,CountyOrParish_San Joaquin,...,PostalCode_95942,PostalCode_95943,PostalCode_95948,PostalCode_95949,PostalCode_95953,PostalCode_95954,PostalCode_95958,PostalCode_95961,PostalCode_95963,PostalCode_95965,PostalCode_95966,PostalCode_95968,PostalCode_95969,PostalCode_95971,PostalCode_95973,PostalCode_95979,PostalCode_95988,PostalCode_95991,PostalCode_95993,PostalCode_96001,PostalCode_96002,PostalCode_96003,PostalCode_96007,PostalCode_96019,PostalCode_96021,PostalCode_96022,PostalCode_96025,PostalCode_96029,PostalCode_96035,PostalCode_96041,PostalCode_96044,PostalCode_96052,PostalCode_96055,PostalCode_96063,PostalCode_96067,PostalCode_96073,PostalCode_96080,PostalCode_96088,PostalCode_96091,PostalCode_96092,PostalCode_96093,PostalCode_96094,PostalCode_96097,PostalCode_96101,PostalCode_96106,PostalCode_96114,PostalCode_96130,PostalCode_96137,PostalCode_96143,PostalCode_96150
0,-0.0058,0.5216,-0.5641,-0.0217,-0.3009,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,1.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.9636,0.5216,1.1989,-0.0205,-0.6978,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,-0.4408,0.5216,-0.5641,-0.0219,0.0599,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0.6260,1.5532,0.3174,-0.0218,1.7558,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0.8652,1.5532,1.1989,-0.0218,1.6836,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 16. Add the target and split labels

The training and test observations are combined into one model-ready dataset. The `split` column allows the later notebooks to preserve the same chronological split.


In [17]:
train_clean = X_train_processed_df.copy()
train_clean[target_col] = y_train.to_numpy()
train_clean["split"] = "train"

test_clean = X_test_processed_df.copy()
test_clean[target_col] = y_test.to_numpy()
test_clean["split"] = "test"

cleaned_model_data = pd.concat(
    [train_clean, test_clean],
    axis=0,
    ignore_index=True,
)

final_missing_count = int(
    cleaned_model_data
    .drop(columns=["split"])
    .isna()
    .sum()
    .sum()
)

if final_missing_count:
    raise ValueError(
        f"{final_missing_count} missing values remain "
        "in the model-ready dataset."
    )

print(
    "Final encoded dataset shape:",
    cleaned_model_data.shape,
)
print(
    "Remaining missing values:",
    final_missing_count,
)
print("\nSplit counts:")
print(
    cleaned_model_data["split"]
    .value_counts()
)

cleaned_model_data.head()


Final encoded dataset shape: (61727, 2449)
Remaining missing values: 0

Split counts:
split
train    49703
test     12024
Name: count, dtype: int64


C:\Users\18327\AppData\Local\Temp\ipykernel_4248\9676438.py:19: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  .sum()


,LivingArea,Bedrooms,Bathrooms,LotSize,YearBuilt,LivingArea_missing,Bedrooms_missing,Bathrooms_missing,LotSize_missing,YearBuilt_missing,CountyOrParish_Alameda,CountyOrParish_Amador,CountyOrParish_Butte,CountyOrParish_Calaveras,CountyOrParish_Colusa,CountyOrParish_Contra Costa,CountyOrParish_El Dorado,CountyOrParish_Foreign Country,CountyOrParish_Fresno,CountyOrParish_Glenn,CountyOrParish_Humboldt,CountyOrParish_Imperial,CountyOrParish_Inyo,CountyOrParish_Kern,CountyOrParish_Kings,CountyOrParish_Lake,CountyOrParish_Lassen,CountyOrParish_Los Angeles,CountyOrParish_Madera,CountyOrParish_Marin,CountyOrParish_Mariposa,CountyOrParish_Mendocino,CountyOrParish_Merced,CountyOrParish_Modoc,CountyOrParish_Mono,CountyOrParish_Monterey,CountyOrParish_Napa,CountyOrParish_Nevada,CountyOrParish_Orange,CountyOrParish_Other,CountyOrParish_Other State,CountyOrParish_Placer,CountyOrParish_Plumas,CountyOrParish_Riverside,CountyOrParish_Sacramento,CountyOrParish_San Benito,CountyOrParish_San Bernardino,CountyOrParish_San Diego,CountyOrParish_San Francisco,CountyOrParish_San Joaquin,...,PostalCode_95948,PostalCode_95949,PostalCode_95953,PostalCode_95954,PostalCode_95958,PostalCode_95961,PostalCode_95963,PostalCode_95965,PostalCode_95966,PostalCode_95968,PostalCode_95969,PostalCode_95971,PostalCode_95973,PostalCode_95979,PostalCode_95988,PostalCode_95991,PostalCode_95993,PostalCode_96001,PostalCode_96002,PostalCode_96003,PostalCode_96007,PostalCode_96019,PostalCode_96021,PostalCode_96022,PostalCode_96025,PostalCode_96029,PostalCode_96035,PostalCode_96041,PostalCode_96044,PostalCode_96052,PostalCode_96055,PostalCode_96063,PostalCode_96067,PostalCode_96073,PostalCode_96080,PostalCode_96088,PostalCode_96091,PostalCode_96092,PostalCode_96093,PostalCode_96094,PostalCode_96097,PostalCode_96101,PostalCode_96106,PostalCode_96114,PostalCode_96130,PostalCode_96137,PostalCode_96143,PostalCode_96150,ClosePrice,split
0,-0.0058,0.5216,-0.5641,-0.0217,-0.3009,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,1.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"1,998,000.0000",train
1,0.9636,0.5216,1.1989,-0.0205,-0.6978,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.0000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"2,214,421.0000",train
2,-0.4408,0.5216,-0.5641,-0.0219,0.0599,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"1,200,000.0000",train
3,0.6260,1.5532,0.3174,-0.0218,1.7558,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"3,100,000.0000",train
4,0.8652,1.5532,1.1989,-0.0218,1.6836,-0.0215,0,-0.0045,-0.1340,-0.0242,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"2,900,000.0000",train


## 17. Save the preprocessing outputs

Both files are saved inside `results/02/`:

1. `cleaned_preprocessed_data.csv` contains the encoded, model-ready features.
2. `cleaned_unencoded_data.csv` contains the cleaned variables before encoding.

The encoded CSV can be large because one-hot encoding creates many columns. Saving may take several minutes.


In [18]:
cleaned_model_data.to_csv(
    results_dir / "cleaned_preprocessed_data.csv",
    index=False
)

model_df.to_csv(
    results_dir / "cleaned_unencoded_data.csv",
    index=False
)

print("Files saved successfully in results/02.")


Files saved successfully in results/02.


## Conclusion

This notebook prepares the residential single-family data for modeling by:

- converting the selected variables to suitable data types;
- removing observations with a missing target;
- creating numerical missing-value indicators;
- assigning the most recent month to the test set;
- fitting median imputation, standardization, and one-hot encoding on the training data only; and
- applying the fitted preprocessing pipeline to the test data.

Generated files:

- `results/02/cleaned_preprocessed_data.csv`
- `results/02/cleaned_unencoded_data.csv`

The later model notebooks should load:

`results/02/cleaned_preprocessed_data.csv`

Before uploading this notebook to GitHub, run all cells from top to bottom and save the notebook so its outputs are embedded. The large encoded CSV normally should not be uploaded to a standard GitHub repository.
